# 🔧 Tool Calling — the manual agent loop

Notebook **040** of the pi.dev series. So far the model has only *talked*. Now we
give it **tools**: functions it can ask us to run. This is the foundation of every
agent.

The loop is simple and worth internalizing:

1. Send the model a `Context` that includes a list of `tools`.
2. The model may reply with `stopReason: "toolUse"` and one or more `toolCall`
   content blocks (name + JSON arguments) instead of a final answer.
3. **We** execute each requested tool locally and append a `toolResult` message.
4. We call the model **again** with the enriched history. It either calls more
   tools or, when satisfied, returns a normal answer with `stopReason: "stop"`.

pi-ai never runs your code for you — it just relays the request and the result.
That transparency is the whole point of this notebook.

> **Kernel:** Deno.

## 1. Load env & register Azure OpenAI

Same bootstrap as every notebook: walk up for a `.env`, then register the Azure
provider via the shared `registerAzure()` helper.

In [ ]:
import { loadEnvUp } from "playground/env";
const env = await loadEnvUp();

In [ ]:
import { registerAzure } from "playground/azure";
const { models, model, modelId } = registerAzure();

## 2. Define the tools

A `Tool` is just `{ name, description, parameters }`, where `parameters` is a
**TypeBox** schema (imported as `Type`). pi-ai serializes that schema to JSON
Schema and sends it to the model; later, `validateToolCall()` checks the
arguments the model sends back against the same schema.

We define two toy tools whose implementations are **local, deterministic
functions** — in a real app these would hit a weather API, a database, the
filesystem, etc.

In [ ]:
import { Type, type Tool, validateToolCall } from "@earendil-works/pi-ai";

// The schemas the MODEL sees. Descriptions matter — they guide when/how the
// model fills each field.
const tools: Tool[] = [
  {
    name: "get_weather",
    description: "Get the current weather forecast for a city.",
    parameters: Type.Object({
      city: Type.String({ description: "City name, e.g. \"Milan\"" }),
    }),
  },
  {
    name: "calculator",
    description: "Do basic arithmetic on two numbers.",
    parameters: Type.Object({
      a: Type.Number(),
      b: Type.Number(),
      op: Type.String({ description: "one of: add, sub, mul, div" }),
    }),
  },
];

// The local IMPLEMENTATIONS. This is where real side effects would live.
function runGetWeather(args: { city: string }): string {
  // Deterministic fake forecast so the notebook is reproducible offline.
  const temps: Record<string, number> = { milan: 24, london: 15, tokyo: 27 };
  const t = temps[args.city.toLowerCase()] ?? 20;
  return `${args.city}: sunny, ${t}°C, light breeze.`;
}

function runCalculator(args: { a: number; b: number; op: string }): string {
  const { a, b, op } = args;
  const result = op === "add" ? a + b
    : op === "sub" ? a - b
    : op === "mul" ? a * b
    : op === "div" ? a / b
    : NaN;
  if (Number.isNaN(result)) return `Unknown op "${op}". Use add, sub, mul, or div.`;
  return `${a} ${op} ${b} = ${result}`;
}

console.log(`Defined ${tools.length} tools: ${tools.map((t) => t.name).join(", ")}`);

## 3. The agent loop

`runAgent()` implements steps 1–4 above. Key details:

- We cap the loop at `MAX_ROUNDS` so a misbehaving model can't spin forever.
- `reply.content.filter((b) => b.type === "toolCall")` pulls the tool requests
  out of the assistant message (which may also contain text/thinking blocks).
- `validateToolCall(tools, call)` returns the **validated, typed arguments** or
  throws. We catch that and feed the error back as a `toolResult` with
  `isError: true`, so the model can see what it got wrong and retry.
- Every tool result is appended to `context.messages` before the next round.

In [ ]:
import { type Context, type Message } from "@earendil-works/pi-ai";

// Running totals across every round-trip in a runAgent() call.
let totalIn = 0;
let totalOut = 0;
let totalCost = 0;

async function runAgent(userText: string): Promise<void> {
  const context: Context = {
    systemPrompt:
      "You are a helpful assistant. Use the provided tools when they help answer " +
      "the question. After gathering tool results, give a concise final answer.",
    messages: [{ role: "user", content: userText, timestamp: Date.now() }],
    tools,
  };

  console.log(`\n🧑 User: ${userText}\n`);

  const MAX_ROUNDS = 5;
  for (let round = 1; round <= MAX_ROUNDS; round++) {
    const reply = await models.completeSimple(model, context);
    context.messages.push(reply);
    totalIn += reply.usage.input;
    totalOut += reply.usage.output;
    totalCost += reply.usage.cost.total;

    const toolCalls = reply.content.filter((b) => b.type === "toolCall");

    // No tool calls → the model produced its final answer.
    if (toolCalls.length === 0) {
      const text = reply.content
        .filter((b) => b.type === "text")
        .map((b) => (b as { text: string }).text)
        .join("");
      console.log(`🤖 Assistant (round ${round}, stop=${reply.stopReason}):\n${text}\n`);
      return;
    }

    console.log(`🔁 Round ${round}: model requested ${toolCalls.length} tool call(s).`);

    for (const call of toolCalls) {
      if (call.type !== "toolCall") continue; // narrow the union for TypeScript
      let resultText: string;
      let isError = false;
      try {
        const args = validateToolCall(tools, call); // throws on schema mismatch
        resultText = call.name === "get_weather"
          ? runGetWeather(args)
          : call.name === "calculator"
          ? runCalculator(args)
          : `Unknown tool: ${call.name}`;
      } catch (err) {
        isError = true;
        resultText = `Invalid arguments: ${(err as Error).message}`;
      }
      console.log(
        `   • ${call.name}(${JSON.stringify(call.arguments)}) → ${resultText}` +
          (isError ? "  ⚠️" : ""),
      );

      const toolResult: Message = {
        role: "toolResult",
        toolCallId: call.id,
        toolName: call.name,
        content: [{ type: "text", text: resultText }],
        isError,
        timestamp: Date.now(),
      };
      context.messages.push(toolResult);
    }
  }

  console.log(`⚠️  Hit MAX_ROUNDS (${MAX_ROUNDS}) without a final answer.`);
}

console.log("runAgent() ready.");

## 4. Run it — a prompt that needs *both* tools

The question below can't be answered from the model's own knowledge: it needs a
live weather lookup **and** an exact multiplication. Watch the loop request both
tools (often in a single round), receive our results, and then compose a final
answer.

In [ ]:
await runAgent("What's the weather in Milan, and what is 47 times 19?");

console.log(
  `\n— totals — tokens: ${totalIn} in / ${totalOut} out  |  cost: $${totalCost.toFixed(6)}`,
);

## ✅ What just happened

1. We defined two `Tool`s with **TypeBox** schemas and matching local functions.
2. `models.completeSimple(model, context)` returned an assistant message whose
   `content` held `toolCall` blocks (`stopReason: "toolUse"`).
3. We validated each call with `validateToolCall()`, executed it, and pushed a
   `toolResult` message back into the conversation.
4. Looping until `stopReason: "stop"` let the model gather everything it needed
   and write the final answer. **That loop is an agent** — everything fancier is
   a refinement of it.

### Things to try
- Send a bad prompt that makes the model call `calculator` with a wrong `op`,
  and watch the `isError` result steer it to retry.
- Add a third tool (e.g. a unit converter) and a prompt that chains all three.

**Next:** notebook **050 — Structured output**, where we use a schema to force the
model to return typed JSON instead of prose.